In [26]:
import numpy as np
import pandas as pd
import ast 
import subprocess
import os

In [27]:
benchmark = pd.read_csv("../Results/heterogeneous_benchmark_results.csv")
benchmark['Same Base_Pair_Counts'] = benchmark['Same Base_Pair_Counts'].map({'True':True, 'False': False}).astype(bool)

benchmark.describe()

,Test_ID,Possibility_ID,Total_Possibilities,m,Soup_MFE,Vienna_MFE,MFE_Diff,Soup_Time(s),Vienna_Time(s),BP_(Vienna - Soup)
count,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000
mean,16.337931,1.634483,2.268966,9.924138,-248.851724,-246.695172,-2.156552,313.367947,0.058105,2.310345
std,10.983081,0.926566,1.237466,5.192252,216.018150,212.501083,10.862121,643.210056,0.061084,6.884481
min,1.000000,1.000000,1.000000,2.000000,-845.200000,-845.200000,-71.800000,0.005368,0.002156,0.000000
25%,7.000000,1.000000,1.000000,5.000000,-336.500000,-336.500000,0.000000,4.409313,0.012288,0.000000
50%,14.000000,1.000000,2.000000,11.000000,-190.700000,-190.700000,0.000000,39.080890,0.037980,0.000000
75%,26.000000,2.000000,3.000000,14.000000,-88.500000,-88.500000,0.000000,263.381100,0.087719,0.000000
max,36.000000,6.000000,6.000000,20.000000,9.800000,9.800000,0.000000,3730.576573,0.282309,50.000000


### Count number of instances with non-zero MFE difference and put to different_benchmark dataframe

In [28]:
non_zero_diff_count = (benchmark['MFE_Diff'] != 0).sum()
different_benchmark = benchmark[benchmark['MFE_Diff'] != 0]
print(f"Number of instances with non-zero MFE difference: {non_zero_diff_count}")

Number of instances with non-zero MFE difference: 9


In [29]:
# save different_benchmark to csv file
different_benchmark.to_csv("../Results/different_MFE_benchmark.csv", index=False)

### Count number of same structure, same shape (different structure), same number of base pairs (different shape) over the total number of instances

In [30]:
num_same = 0
num_same_shape = 0 # different structure
num_same_base_pairs = 0 # different shape
for index, row in benchmark.iterrows():
    if row['Struct_Match_Type'] == "Same structure":
        num_same += 1
    elif row['Struct_Match_Type'] == "Same with inverse" or row['Struct_Match_Type'] == "Same with shift" or row['Struct_Match_Type'] == "Same with inverse shift":
        num_same_shape += 1
    elif row['Same Base_Pair_Counts'] and not row['Struct_Match']:
        num_same_base_pairs += 1
total_instances = len(benchmark)

print(f"Number of instances with same structure: {num_same} ({num_same/total_instances:.2%})")
print(f"Number of instances with same shape but different structure: {num_same_shape} ({num_same_shape/total_instances:.2%})")
print(f"Number of instances with same number of base pairs but different shape: {num_same_base_pairs} ({num_same_base_pairs/total_instances:.2%})")
print(f"Number of completely different base pairs: {total_instances - num_same - num_same_shape - num_same_base_pairs} ({(total_instances - num_same - num_same_shape - num_same_base_pairs)/total_instances:.2%})")

Number of instances with same structure: 83 (57.24%)
Number of instances with same shape but different structure: 7 (4.83%)
Number of instances with same number of base pairs but different shape: 31 (21.38%)
Number of completely different base pairs: 24 (16.55%)


### Pick some different instances to plot

In [38]:
# we pick all instances of different structure, 
# and 3 that are assumed to have the same structure but needs to be shifted or inverted
chosen_instances = benchmark.copy()
list_matches = ["Same with inverse", "Same with shift", "Same with inverse shift"]
for index, row in chosen_instances.iterrows():
    match = row['Struct_Match_Type']
    if match == "False":
        continue
    if match in list_matches:
        list_matches.remove(row['Struct_Match_Type'])
    else:
        chosen_instances.drop(index=index, inplace=True)
# chosen_instances        

In [34]:
# save chosen_instances to csv file
chosen_instances.to_csv("../Results/filtered_hetero.csv", index=False)

# Plot and visulize

In [12]:
# benchmark = pd.read_csv("../Results/filtered_hetero.csv") 
# change the above csv file to the file of interest if needed

output_file = "../Plots/structures.fold"
with open(output_file, "w") as f:
    for index, row in benchmark.iterrows():
        test_id = row['Test_ID']
        possibility_id = row['Possibility_ID']
        sequence = row['Full_Constructed_Seq']
        soup_struct = row['Structure_Soup']
        vienna_struct = row['Structure_Vienna']
        
        f.write(f">{test_id}_{possibility_id}_Soup_fold\n")
        f.write(f"{sequence}\n")
        f.write(f"{soup_struct}\n")
        
        f.write(f">{test_id}_{possibility_id}_Vienna\n")
        f.write(f"{sequence}\n")
        f.write(f"{vienna_struct}\n")

In [14]:
file_name = "structures.fold"
output_dir = "../Plots/eps"

RNAplot_proc = subprocess.run(
        ["RNAplot", "--pre=/paircolor {1 0 0 setrgbcolor} bind def"],
        stdin=open(f"../Plots/{file_name}", "r"),
        cwd=output_dir,
        capture_output=True, 
        text=True
    )

### Next step
When the eps files are already in the Plots/Eps folder, run the following commands in the `Scripts` folder to convert all `.eps` files into `.pdf` files:

```bash
chmod u+x pdf_converter.sh
./pdf_converter.sh
```

## Split the filterd instances into different types then plot to see more easily

In [40]:
# split chosen_benchmark into 3 dataframes: different number of base pairs, same base pairs but different structure, and the rest
# chosen_instances = pd.read_csv("results/chosen_benchmark.csv")

different_base_pairs = chosen_instances[chosen_instances['Same Base_Pair_Counts'] == False]
same_base_pairs = chosen_instances[(chosen_instances['Same Base_Pair_Counts'] == True) & (chosen_instances['Struct_Match'] == False)]
assume_same = chosen_instances[(chosen_instances['Same Base_Pair_Counts'] == True) & (chosen_instances['Struct_Match'] == True)]
different_MFE_benchmark = chosen_instances[chosen_instances['MFE_Diff'] != 0]


In [ ]:
# save the 4 dataframes to csv files

different_MFE_benchmark.to_csv("../Results/different_MFE_benchmark.csv", index=False)
different_base_pairs.to_csv("../Results/different_base_pairs_benchmark.csv", index=False)
same_base_pairs.to_csv("../Results/same_base_pairs_benchmark.csv", index=False)
assume_same.to_csv("../Results/assume_same_benchmark.csv", index=False)

## Plot and visulize specific instances

In [46]:
output_dir = "../Plots/specific"
# Automatically create the specific subfolder if it does not exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs("../Plots/specific/eps", exist_ok=True) # Prepare the eps folder 
os.makedirs("../Plots/specific/pdf", exist_ok=True) # Prepare the pdf folder

target_benchmark = assume_same
# target_benchmark = pd.read_csv("../Results/filtered_hetero.csv") 
# change the above csv file to the file of interest if needed

output_file = "../Plots/specific/structures_specific.fold"
with open(output_file, "w") as f:
    for index, row in target_benchmark.iterrows():
        test_id = row['Test_ID']
        possibility_id = row['Possibility_ID']
        sequence = row['Full_Constructed_Seq']
        soup_struct = row['Structure_Soup']
        vienna_struct = row['Structure_Vienna']
        
        f.write(f">{test_id}_{possibility_id}_Soup_fold\n")
        f.write(f"{sequence}\n")
        f.write(f"{soup_struct}\n")
        
        f.write(f">{test_id}_{possibility_id}_Vienna\n")
        f.write(f"{sequence}\n")
        f.write(f"{vienna_struct}\n")

In [47]:
file_name = "structures_specific.fold"
output_dir = "../Plots/specific/eps"

RNAplot_proc = subprocess.run(
        ["RNAplot", "--pre=/paircolor {1 0 0 setrgbcolor} bind def"],
        stdin=open(f"../Plots/specific/{file_name}", "r"),
        cwd=output_dir,
        capture_output=True, 
        text=True
    )

### Next step
When the eps files are already in the Plots/Eps folder, run the following commands in the `Scripts` folder to convert all `.eps` files into `.pdf` files:

```bash
chmod u+x pdf_converter.sh
./pdf_converter.sh ../Plots/specific
```

# See which has more base pairs in different_base_pairs_benchmark

In [ ]:
# different_base_pairs = pd.read_csv("Results/different_base_pairs_benchmark.csv")    
different_base_pairs['BP_(Vienna - Soup)'] = 0 # init with integers 
for index, row in different_base_pairs.iterrows():
    if row['Struct_Match']:
        continue
    soup_counts = ast.literal_eval(row['Base_Pair_Counts_Soup'])
    vienna_counts = ast.literal_eval(row['Base_Pair_Counts_Vienna'])     
    # if soup_counts == 0 or vienna_counts == 0:
    #     print(f"Row {index}: One or both base pair counts are zero.")
    #     print(row['Test_ID'])
    #     print(row['Possibility_ID'])
    #     print(row['Base_Pair_Counts_Soup'])
    #     print(row['Base_Pair_Counts_Vienna'])
    #     print(f"Soup counts: {soup_counts}, Vienna counts: {vienna_counts}")
    soup_total = sum(soup_counts.values())
    vienna_total = sum(vienna_counts.values())
    different_base_pairs.at[index, 'BP_(Vienna - Soup)'] = vienna_total - soup_total

In [ ]:
different_base_pairs.to_csv("../Results/different_base_pairs_benchmark.csv", index=False)

## Reset the dataframe index of Test_ID if needed

In [ ]:
# rewrite the test_ID in chosen_benchmark to be from 1 to len(chosen_benchmark)
start = 1
actual_id = 1
aux_count = 0 
# counter for cases of identical test_id but also identical_poss_id, meaning they are two different test cases

benchmark = pd.read_csv("../Results/filtered_hetero.csv") # choose csv file to rewrite test_ID
for index, row in benchmark.iterrows():
    if actual_id == row['Test_ID'] and aux_count != row['Possibility_ID']:
        benchmark.at[index, 'Test_ID'] = start
        aux_count += 1
    else: # next test case
        aux_count = 1
        start += 1
        actual_id = row['Test_ID']
        benchmark.at[index, 'Test_ID'] = start
benchmark.to_csv("../Results/patched_hetero_benchmark.csv", index=False)